# Render LOTE (escalonado) · RESTO del lote
Ctrl+F9. Compuerta GPU (nunca CPU) -> dependencias una vez -> por episodio: clonar `colab/<ep>`, `npm ci`, render por partes reanudables, loudnorm, descarga `<ep>.mp4`.

In [ ]:
import subprocess
_g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, text=True, capture_output=True, executable="/bin/bash")
GPU = (_g.stdout.strip().splitlines() or [""])[0]
if _g.returncode != 0 or not GPU:
    raise SystemExit("SIN GPU: lote CANCELADO (regla: nunca CPU).")
print("GPU OK:", GPU)

In [ ]:
PROYECTOS = ["religioso-infantil-06-jose-tunica", "religioso-infantil-07-moises-cesta", "religioso-infantil-08-oveja-perdida"]
REPO = "jonisral2211-dev/render-remotion"
CONCURRENCIA_GPU = 2
PARTES_S = 8   # 0 = un solo render; >0 = trozos de ~N s con cortes en limites de escena (reanudable)

In [ ]:
import subprocess, os, glob, json, shutil, time
def sh(c, check=True):
    r = subprocess.run(c, shell=True, text=True, executable="/bin/bash")
    if check and r.returncode != 0:
        raise RuntimeError(f"Fallo ({r.returncode}): {c}")
    return r.returncode == 0
def out(c):
    return subprocess.run(c, shell=True, text=True, capture_output=True, executable="/bin/bash").stdout.strip()
assert out("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null"), "SIN GPU"
SUDO = "" if os.geteuid() == 0 else "sudo "
sh(f"{SUDO}apt-get update -qq")
libs = "ffmpeg libnss3 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libvulkan1 mesa-vulkan-drivers unzip git"
sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {libs}")
sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || {SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2")
nv = out("node -v 2>/dev/null")
if not nv.startswith(("v20", "v22", "v24")):
    sh(f"curl -fsSL https://deb.nodesource.com/setup_20.x | {SUDO}bash - && {SUDO}apt-get install -y -qq nodejs")
print("node", out("node -v"), "|", out("ffmpeg -version | head -1"))

In [ ]:
BASE = "/content" if os.path.isdir("/content") else "/kaggle/working"
os.chdir(BASE)
RES = {}
def trozos(props, parte_s):
    """Rangos de fotogramas [a, b] (inclusive) con cortes en limites de escena; ~parte_s segundos cada uno."""
    P = json.load(open(props)); fps = P["fps"]; N = P["durationInFrames"]
    if not parte_s:
        return [(0, N - 1)]
    cortes = sorted({round(e["start"] * fps) for e in P["escenas"] if 0 < round(e["start"] * fps) < N})
    rangos, a = [], 0
    for c in cortes + [N]:
        if c - a >= parte_s * fps or c == N:
            rangos.append((a, c - 1)); a = c
    return rangos
for P in PROYECTOS:
    print("=" * 20, P)
    os.chdir(BASE)
    final_prev = f"{BASE}/proy_{P}/salida/{P}.mp4"
    if os.path.exists(final_prev):
        print("ya renderizado, se salta:", final_prev); RES[P] = final_prev; continue
    shutil.rmtree(f"proy_{P}", ignore_errors=True)
    sh(f"git clone -q --depth 1 --branch colab/{P} https://github.com/{REPO}.git proy_{P}")
    props = glob.glob(f"proy_{P}/render_jobs/*/props.json")
    assert props, f"{P}: sin props.json"
    PROPS = os.path.abspath(props[0])
    os.chdir(f"{BASE}/proy_{P}/remotion-studio")
    sh("npm ci --silent --no-audit --no-fund")
    sh("npx remotion browser ensure")
    assert out("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null"), "SIN GPU: render CANCELADO (nunca CPU)"
    os.makedirs("../salida", exist_ok=True)
    rangos = trozos(PROPS, PARTES_S)
    print("partes:", rangos)
    usado, t = None, time.time()
    for k, (fa, fb) in enumerate(rangos):
        destino = f"../salida/parte_{k:02d}.mp4"
        if os.path.exists(destino) and os.path.getsize(destino) > 10000:
            print("parte ya hecha:", destino); continue   # reanudable: no se repite lo ya renderizado
        extra = "" if len(rangos) == 1 else f"--frames={fa}-{fb}"
        ok = False
        for gl in ([usado] if usado else ["angle", "webgl"]):
            if sh(f'npx remotion render src/index.ts Main {destino} --props="{PROPS}" --codec=h264 --gl={gl} --concurrency={CONCURRENCIA_GPU} {extra} --log=info', check=False):
                usado, ok = gl, True; break
            print(f"--gl={gl} fallo, probando el siguiente...")
        assert ok, f"{P}: render con GPU fallo en la parte {k} (no se cae a CPU)"
    if len(rangos) == 1:
        sh("cp ../salida/parte_00.mp4 ../salida/crudo.mp4")
    else:
        with open("../salida/partes.txt", "w") as fh:
            fh.write("".join(f"file 'parte_{k:02d}.mp4'\n" for k in range(len(rangos))))
        sh("cd ../salida && ffmpeg -y -loglevel error -f concat -safe 0 -i partes.txt -c copy crudo.mp4")
    final = f"../salida/{P}.mp4"
    LN = "loudnorm=I=-14:TP=-1.5:LRA=11"
    vc = "-c:v h264_nvenc -preset p5 -cq 21" if sh("ffmpeg -hide_banner -encoders 2>/dev/null | grep -q h264_nvenc", check=False) else "-c:v copy"
    if not sh(f'ffmpeg -y -loglevel error -i ../salida/crudo.mp4 {vc} -af {LN} -c:a aac -b:a 192k {final}', check=False):
        sh(f'ffmpeg -y -loglevel error -i ../salida/crudo.mp4 -c:v copy -af {LN} -c:a aac -b:a 192k {final}')
    RES[P] = os.path.abspath(final)
    print(f"Render OK {P}: {time.time()-t:.0f} s (gl={usado}, partes={len(rangos)})")
    sh(f"ls -lh {final}; ffprobe -v error -show_entries format=duration -of csv=p=0 {final}")
    try:
        from google.colab import files; files.download(RES[P])
    except ImportError:
        print("Kaggle: queda en", RES[P], "(panel Output)")
print("LOTE TERMINADO:", list(RES))